# Deploy NER model pipeline by using an online endpoint 

Create online endpoint for the NER model pipeline, don't have to create and manage the underlying infrastructure in ML Studio.

Managed online endpoints help to deploy your ML models in a turnkey manner. Managed online endpoints work with powerful CPU and GPU machines in Azure in a scalable, fully managed way. Managed online endpoints take care of serving, scaling, securing, and monitoring your models, freeing you from the overhead of setting up and managing the underlying infrastructure. 

For more information, see [What are Azure Machine Learning endpoints?](https://learn.microsoft.com/azure/machine-learning/concept-endpoints), and [Deploy an ML model with an online endpoint](https://learn.microsoft.com/azure/machine-learning/how-to-deploy-online-endpoints).

## Prerequisites

* To use Azure Machine Learning, you must have an Azure subscription. If you don't have an Azure subscription, create a free account before you begin. Try the [free or paid version of Azure Machine Learning](https://azure.microsoft.com/free/).

* Install and configure the [Python SDK v2](sdk/setup.sh).

* You must have an Azure resource group, and you (or the service principal you use) must have Contributor access to it.

* You must have an Azure Machine Learning workspace. 

# 1. Connect to Azure Machine Learning Workspace

The [workspace](https://docs.microsoft.com/en-us/azure/machine-learning/concept-workspace) is the top-level resource for Azure Machine Learning, providing a centralized place to work with all the artifacts you create when you use Azure Machine Learning. In this section we will connect to the workspace in which the job will be run.

## 1.1. Import the required libraries

In [43]:
# import required libraries
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    CodeConfiguration,
    OnlineRequestSettings
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

## 1.2. Configure workspace details and get a handle to the workspace

To connect to a workspace, we need identifier parameters - a subscription, resource group and workspace name. We will use these details in the `MLClient` from `azure.ai.ml` to get a handle to the required Azure Machine Learning workspace. We use the default [default azure authentication](https://docs.microsoft.com/en-us/python/api/azure-identity/azure.identity.defaultazurecredential?view=azure-python) for this tutorial. Check the [configuration notebook](../../jobs/configuration.ipynb) for more details on how to configure credentials and connect to a workspace.

In [44]:
# enter details of your AML workspace
subscription_id = "710c48d7-7060-4d97-9be0-699f76c25447"
resource_group = "rg-gst-dev-ussc-01"
workspace = "ml-gst-dev-usscc-01"

In [45]:
# get a handle to the workspace
ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace
)

## 1.3. Register the model
Register the model when any of the files in dependencies folder is updated. This will create a new version of the model "gst-ner-model" in Azure ML Studo Model registry.

In [46]:
model_name = "gst-gpt-ner-model"

In [47]:
# #Register the model on Azure
# from azureml.core import Workspace, Model

# # Assuming you already have your Workspace set up
# ws = Workspace.from_config()

# model = Model.register(workspace=ws,
#                        model_name=model_name,
#                        model_path="./dependencies",
#                        description="GST NER model with dependencies")


# #############################################################
# # whenever we have a change in depenency folder ######
# ##############################################

# 2. Define endpoint and deployment

## 2.1 Define the endpoint

To define an endpoint, you need to specify:

* Endpoint name: The name of the endpoint. It must be unique in the Azure region. For more information on the naming rules, see [managed online endpoint limits](how-to-manage-quotas.md#azure-machine-learning-managed-online-endpoints).
* Authentication mode: The authentication method for the endpoint. Choose between key-based authentication and Azure Machine Learning token-based authentication. A key doesn't expire, but a token does expire. For more information on authenticating, see [Authenticate to an online endpoint](how-to-authenticate-online-endpoint.md).
* Optionally, you can add a description and tags to your endpoint.

In [48]:
# Define an endpoint name
endpoint_name = "gst-gpt-ner-endpoint-dev-public"
# endpoint_name = "gst-ner-endpoint-dev"

# Example way to define a random name
import datetime

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="GST NER Model endpoint",
    auth_mode="key",
    public_network_access="disabled",
    tags={"Model": "GPT NER"},
)

request_settings = OnlineRequestSettings(max_concurrent_requests_per_instance= 100)

## 2.2 Define the deployment

A deployment is a set of resources required for hosting the model that does the actual inferencing. To deploy a model, you must have:

- Model files (or the name and version of a model that's already registered in your workspace).
- A scoring script, that is, code that executes the model on a given input request. The scoring script receives data submitted to a deployed web service and passes it to the model. The script then executes the model and returns its response to the client. The scoring script is specific to your model and must understand the data that the model expects as input and returns as output. In this example, we have a *score.py* file.
- An environment in which your model runs. The environment can be a Docker image with Conda dependencies or a Dockerfile.
- Settings to specify the instance type and scaling capacity.

The following table describes the key attributes of a deployment:

| Attribute      | Description                                                                                                                                                                                                                                                                                                                                                                                    |
|-----------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Name           | The name of the deployment.                                                                                                                                                                                                                                                                                                                                                                    |
| Endpoint name  | The name of the endpoint to create the deployment under.                                                                                                                                                                                                                                                                                                                                       |
| Model          | The model to use for the deployment. This value can be either a reference to an existing versioned model in the workspace or an inline model specification. Make sure to update the model version after registering it.                                                                                                                                                                                                                                   |
| Code path      | The path to the directory on the local development environment that contains all the Python source code for scoring the model. You can use nested directories and packages.                                                                                                                                                                                                                    |
| Scoring script | The relative path to the scoring file in the source code directory. This Python code must have an `init()` function and a `run()` function. The `init()` function will be called after the model is created or updated (you can use it to cache the model in memory, for example). The `run()` function is called at every invocation of the endpoint to do the actual scoring and prediction. |
| Environment    | The environment to host the model and code. This value can be either a reference to an existing versioned environment in the workspace or an inline environment specification.                                                                                                                                                                                                                 |
| Instance type  | The VM size to use for the deployment. For the list of supported sizes, see [Managed online endpoints SKU list](reference-managed-online-endpoints-vm-sku-list.md).                                                                                                                                                                                                                            |
| Instance count | The number of instances to use for the deployment. Base the value on the workload you expect. For high availability, we recommend that you set the value to at least `3`. We reserve an extra 20% for performing upgrades. For more information, see [managed online endpoint quotas](how-to-manage-quotas.md#azure-machine-learning-managed-online-endpoints).                                |

In [49]:
# env = Environment(
#     name="gst-ner-env",
#     conda_file="./environment/conda.yaml",
#     image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
# )

# ml_client.environments.create_or_update(env)

In [50]:
# from azureml.core import Environment

# env = Environment.from_conda_specification(name="test-env", 
#                         file_path="./environment/conda.yaml")
# env.docker.enabled = True
# env.docker.base_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"

In [51]:
model = ml_client.models.get(name=model_name, version=7)
env = Environment(
    name="gst-ner-env",
    conda_file="./environment/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    request_settings=request_settings,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(
        code="./onlinescoring", scoring_script="score.py"
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
    egress_public_network_access="enabled"
)

# 3. Deploy your online endpoint to Azure
Next, deploy your online endpoint to Azure.

## 3.1 Create the endpoint
Using the `endpoint` we defined earlier and the `MLClient` created earlier, we'll now create the endpoint in the workspace. This command will start the endpoint creation and return a confirmation response while the endpoint creation continues.

In [52]:
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc268949dc0>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26898b4f0>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26897e130>: Failed to establish a new connection: [Errno -2] Name or service not known')).


ManagedOnlineEndpoint({'public_network_access': 'Disabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://gst-gpt-ner-endpoint-dev-public.southcentralus.inference.ml.azure.com/score', 'openapi_uri': 'https://gst-gpt-ner-endpoint-dev-public.southcentralus.inference.ml.azure.com/swagger.json', 'name': 'gst-gpt-ner-endpoint-dev-public', 'description': 'GST NER Model endpoint', 'tags': {'Model': 'GPT NER', 'Application': 'GST', 'CostCenter': '1125500', 'DataSensitivity': '', 'Department': '', 'DRTier': '', 'Environment': '', 'Function': '', 'ManagedBy': '', 'OwnedBy': '', 'ProjectID': ''}, 'properties': {'azureml.onlineendpointid': '/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/resourcegroups/rg-gst-dev-ussc-01/providers/microsoft.machinelearningservices/workspaces/ml-gst-dev-usscc-01/onlineendpoints/gst-gpt-ner-endpoint-dev-public', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/providers/Microsoft.MachineLearn

## 3.2 Create the deployment

Using the `blue_deployment` that we defined earlier and the `MLClient` we created earlier, we'll now create the deployment in the workspace. This command will start the deployment creation and return a confirmation response while the deployment creation continues.

In [53]:
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

Check: endpoint gst-gpt-ner-endpoint-dev-public exists
Uploading onlinescoring (0.09 MBs): 100%|██████████| 92512/92512 [00:00<00:00, 886035.85it/s]


Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26811a550>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26808b1f0>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib

............................................

ManagedOnlineDeployment({'private_network_connection': None, 'provisioning_state': 'Succeeded', 'endpoint_name': 'gst-gpt-ner-endpoint-dev-public', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {'Application': 'GST', 'CostCenter': '1125500', 'DataSensitivity': '', 'Department': '', 'DRTier': '', 'Environment': '', 'Function': '', 'ManagedBy': '', 'OwnedBy': '', 'ProjectID': ''}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/providers/Microsoft.MachineLearningServices/locations/southcentralus/mfeOperationsStatus/od:e3d05e55-9be9-4bc9-a48a-257b67f56289:311b86da-07de-4f82-b773-74800350a345?api-version=2023-04-01-preview'}, 'print_as_yaml': True, 'id': '/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/resourceGroups/rg-gst-dev-ussc-01/providers/Microsoft.MachineLearningServices/workspaces/ml-gst-dev-usscc-01/onlineEndpoints/gst-gpt-ner-endpoint-dev-public/deployments/blue', 'Resource__source_pa

In [54]:
# blue deployment takes 100 traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc268118cd0>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc2680ca520>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26898b6d0>: Failed to establish a new connection: [Errno -2] Name or service not known')).


ManagedOnlineEndpoint({'public_network_access': 'Disabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://gst-gpt-ner-endpoint-dev-public.southcentralus.inference.ml.azure.com/score', 'openapi_uri': 'https://gst-gpt-ner-endpoint-dev-public.southcentralus.inference.ml.azure.com/swagger.json', 'name': 'gst-gpt-ner-endpoint-dev-public', 'description': 'GST NER Model endpoint', 'tags': {'Model': 'GPT NER', 'Application': 'GST', 'CostCenter': '1125500', 'DataSensitivity': '', 'Department': '', 'DRTier': '', 'Environment': '', 'Function': '', 'ManagedBy': '', 'OwnedBy': '', 'ProjectID': ''}, 'properties': {'azureml.onlineendpointid': '/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/resourcegroups/rg-gst-dev-ussc-01/providers/microsoft.machinelearningservices/workspaces/ml-gst-dev-usscc-01/onlineendpoints/gst-gpt-ner-endpoint-dev-public', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/providers/Microsoft.MachineLearn

# 4. Test the endpoint with sample data
Using the `MLClient` created earlier, we will get a handle to the endpoint. The endpoint can be invoked using the `invoke` command with the following parameters:
- `endpoint_name` - Name of the endpoint
- `request_file` - File with request data
- `deployment_name` - Name of the specific deployment to test in an endpoint

We will send a sample request using a [json](./model-1/sample-request.json) file. 

In [55]:
# test the blue deployment with some sample data
ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name="blue",
    request_file="./sample-request.json",
)

'{"userInput": {"searchQuery": "gf30, high density"}, "modelOutput": {"entities": {"GRADE": [], "APPLICATION": [], "BRAND": [], "POLYMER": [], "PROPERTY": [], "FILLER": [], "FEATURE": [], "PROCESSING": [], "DELIVERY_FORM": [], "COMPETITOR_GRADE": [], "AUTO_CERT": [], "RAILWAY_CERT": [], "WATER_CERT": [], "NSF_CERT": []}, "unidentified": "gf30, high density"}, "modelVersion": "GPTv12_04_19_24", "apiVersion": "v2.0.0"}'

Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26808b820>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc268118dc0>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fc26811a1f0>: Failed to establish a new connection: [Errno -2] Name or service not known')).


# 5. Managing endpoints and deployments

## 5.1 Get details of the endpoint

In [ ]:
# Get the details for online endpoint
endpoint = ml_client.online_endpoints.get(name=endpoint_name)

# existing traffic details
print(endpoint.traffic)

# Get the scoring URI
print(endpoint.scoring_uri)

## 5.2 Get the logs for the new deployment
Get the logs for the green deployment and verify as needed

In [ ]:
ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=endpoint_name, lines=50
)

# 6. Delete the endpoint


In [13]:
#ml_client.online_endpoints.begin_delete(name=endpoint_name)

# Test

In [13]:
import urllib.request
import json
import os
import ssl

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True) # this line is needed if you use self-signed certificate in your scoring service.

# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {"data":"bmw bonnet"}

body = str.encode(json.dumps(data))

url = 'https://gst-ner-endpoint-dev.southcentralus.inference.ml.azure.com/score'
# Replace this with the primary/secondary key or AMLToken for the endpoint
api_key = '<AML_ENDPOINT_KEY_DEV>'
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

# The azureml-model-deployment header will force the request to go to a specific deployment.
# Remove this header to have the request observe the endpoint traffic rules
headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ api_key), 'azureml-model-deployment': 'blue' }

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

The request failed with status code: 408
content-type: text/plain
date: Thu, 25 Apr 2024 09:43:54 GMT
server: azureml-frontdoor
azureml-model-deployment: blue
content-length: 220
x-request-id: e9b0da70-82b1-4d89-96c4-530a0153bd27
connection: close


upstream request timeout
Please check this guide to understand why this error code might have been returned 
https://docs.microsoft.com/en-us/azure/machine-learning/how-to-troubleshoot-online-endpoints#http-status-codes



In [19]:
!pip install openai==0.28.1

     |████████████████████████████████| 76 kB 2.1 MB/s eta 0:00:01


In [8]:
import openai
openai.api_type = "azure"
openai.api_base = "https://oai-gst-d-usnc-01.openai.azure.com/"
openai.api_version = "2023-07-01-preview"
openai.api_key = "4b216a50a8704d0d9a234c4e4d25b545"

In [9]:
content = "Act as an NER model trained on the data corpus of 'Celanese' which is a global chemical leader in the production of differentiated chemistry solutions and specialty materials used in most major industries and consumer applications. Ensure the output is a structured dictionary format."
fine_tuned_model_id = "gpt-35-turbo-0613: ftjob-1b394e8df8ac44e4af410c3d82604bc6-NERv12"
engine = "oai-gpt35-NERv12-gst-d-usnc-01"
def get_ner_gpt35(query):
    completion = openai.ChatCompletion.create(
    engine=engine,
    seed=12,
    temperature = 0.2,
    model=fine_tuned_model_id,
#         response_format={ "type": "json_object" }
    messages=[
        {"role": "system", "content": content}, 
        {"role": "user", "content": query},
        ],
    )
    return completion.choices[0].message['content']

In [10]:
get_ner_gpt35("high density")

"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': [], 'PROPERTY': [{'property_name': 'density', 'modifier': {'value': 'high', 'min': None, 'max': None, 'unit': ''}, 'property_type': 'property'}], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': []}"

In [61]:
import urllib.request
import json
import os
import ssl

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True) # this line is needed if you use self-signed certificate in your scoring service.

# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {"data":"auto approvel"}

body = str.encode(json.dumps(data))

url = 'https://gst-gpt-ner-endpoint-dev-public.southcentralus.inference.ml.azure.com/score'
# Replace this with the primary/secondary key or AMLToken for the endpoint
api_key = '<AML_ENDPOINT_KEY_DEV_PUBLIC>'
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

# The azureml-model-deployment header will force the request to go to a specific deployment.
# Remove this header to have the request observe the endpoint traffic rules
headers = {'Content-Type':'application/json', 'Authorization':('Bearer '+ api_key), 'azureml-model-deployment': 'blue' }

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"userInput": {"searchQuery": "auto approvel"}, "modelOutput": {"entities": {"GRADE": [], "APPLICATION": [], "BRAND": [], "POLYMER": [], "PROPERTY": [], "FILLER": [], "FEATURE": [], "PROCESSING": [], "DELIVERY_FORM": [], "COMPETITOR_GRADE": [], "AUTO_CERT": [{"oem": "all", "certs": ["all"]}], "RAILWAY_CERT": [], "WATER_CERT": [], "NSF_CERT": []}, "unidentified": ""}, "modelVersion": "GPTv12_04_19_24", "apiVersion": "v2.0.0"}'
